# Dining out spending habits

This is a preliminary analysis of a dataset from a collection of interviews with randomly selected public members about their dining out habits.

The interviews were recorded. The dataset is a sample of data extracted from the interviews.

The orginal dataset is not well formatted and requires some cleaning before processing.

The code is an example how to clean the dataset and creates a sample plot.

In [14]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import altair as alt # Import Altair for plotting

df = pd.read_csv("dining_out_survey.csv")

# Clean the 'Amount spent' column to remove currency symbols; convert to numeric
df['Amount spent'] = df['Amount spent'].astype(str).str.replace('$', '', regex=False).str.replace('€', '', regex=False).str.replace(',', '', regex=False)
df['Amount spent'] = pd.to_numeric(df['Amount spent'], errors='coerce')
# Drop rows where Amount spent could not be converted to a number to avoid errors.
df.dropna(subset=['Amount spent'], inplace=True)

# The column 'Capture Date' which contains the date of the interview may not be well formatted.
# Get all unique date formats, ignore leading/trailing whitespace
date_formats = df['Capture Date'].astype(str).str.strip().unique()
parsed_dates = {}
for fmt in ['%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d']:  # Common formats
    try:
        parsed_dates[fmt] = pd.to_datetime(date_formats, format=fmt, errors='coerce').notna().all()
    except ValueError:
        parsed_dates[fmt] = False

# Find the most common valid format
valid_formats = {fmt: count for fmt, count in parsed_dates.items() if count}
most_common_format = max(valid_formats, key=valid_formats.get, default='%Y-%m-%d')  # Default to '%Y-%m-%d' if no clear winner

# Standardize dates to the most common format
df['Capture Date'] = pd.to_datetime(df['Capture Date'].astype(str).str.strip(), format=most_common_format, errors='coerce')

# Drop rows with invalid dates
df.dropna(subset=['Capture Date'], inplace=True)

# Aggregate total purchases by date
df_agg = df.groupby('Capture Date')['Amount spent'].sum().reset_index()

# Resample to monthly frequency and sum total purchases
df_agg_monthly = df_agg.set_index('Capture Date').resample('ME')['Amount spent'].sum().reset_index()


# The data was collected at different times of the year (whithin the same year).
# The code below creates a plot which explores changes in the amount spent on dining out

# Calculate month-over-month percentage change
df_agg_monthly['Month-over-month percentage change'] = df_agg_monthly['Amount spent'].pct_change() * 100

# Identify significant increases/decreases (exceeding 1 std dev from mean)
mean_change = df_agg_monthly['Month-over-month percentage change'].mean()
std_dev_change = df_agg_monthly['Month-over-month percentage change'].std()
df_agg_monthly['Significant_Change'] = (df_agg_monthly['Month-over-month percentage change'] > mean_change + std_dev_change) | (df_agg_monthly['Month-over-month percentage change'] < mean_change - std_dev_change)

# Create the chart
chart = alt.Chart(df_agg_monthly).encode(
    x=alt.X('Capture Date', axis=alt.Axis(format='%Y-%m-%d')),
    y=alt.Y('Amount spent')
)

# Add the line for total purchases
line = chart.mark_line(point=True).encode()

final_chart = (line).properties(title='Total Amount spent on dining out over time')
#show the chart
final_chart

alt.Chart(...)